In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from catboost import CatBoostRegressor
from collections import deque

In [ ]:
loocback = 5
forward = 5

model = YOLO(r'C:\Users\user\projectolymp\runs\detect\models\train\weights\best.pt')
cap = cv2.VideoCapture(0)
track_history = {}
dataset_rows = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: 
        break
    results = model.track(frame, persist=True, tracker="bytetrack.yaml")
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            if track_id not in track_history:
                track_history[track_id] = deque(maxlen=loocback + forward + 1)
            track = track_history[track_id]
            track.append((float(x), float(y)))
            for i in range(1, len(track)):
                cv2.line(frame, (int(track[i-1][0]), int(track[i-1][1])), 
                         (int(track[i][0]), int(track[i][1])), (0, 0, 255), 2)
            if len(track) >= (loocback + forward):
                past_pts = list(track)[-(loocback + forward):-forward] 
                future_pt = track[-1] 
                dist = np.linalg.norm(np.array(past_pts[0]) - np.array(past_pts[-1]))
                if dist > 10:
                    row = []
                    for i in range(1, len(past_pts)):
                        row.extend([past_pts[i][0] - past_pts[i-1][0], 
                                    past_pts[i][1] - past_pts[i-1][1]])
                    target_dx = future_pt[0] - past_pts[-1][0]
                    target_dy = future_pt[1] - past_pts[-1][1]
                    dataset_rows.append(row + [target_dx, target_dy])
    cv2.imshow("MMS", frame)

    if cv2.waitKey(1) & 0xFF == ord("1"):
        break
cap.release()
cv2.destroyAllWindows()

data = np.array(dataset_rows)
X, Y = data[:, :-2], data[:, -2:]
cb_model = CatBoostRegressor(
    iterations=2000, 
    loss_function='MultiRMSE', 
    depth=6, 
    learning_rate=0.05,
    verbose=100
)
    
cb_model.fit(X, Y)
cb_model.save_model("catboost.cbm")